In [ ]:
import pandas as pd
import pickle
from tqdm import tqdm
import warnings
import gensim
from gensim import corpora, models
from gensim.corpora import Dictionary
warnings.filterwarnings('ignore')
from gensim.models import CoherenceModel #coherence 모델 라이브러리
import matplotlib.pyplot as plt #그래프 그리기
import numpy as np
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, dendrogram
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis

In [ ]:
# 클러스터링 파일 가져오기
with open('DX_Actor.pkl', 'rb') as f:
    df = pickle.load(f)

In [ ]:
df_cluster0 = df[df['cluster'] == 0]
df_cluster0.head()
len(df_cluster0)

In [ ]:
# 단어 사전 만들기
all_documents = list(df_cluster0['tagged_content'])
dictionary = Dictionary(all_documents)
dictionary.token2id

In [ ]:
corpus = []

for doc in all_documents :
    corpus.append(dictionary.doc2bow(doc)) 

In [ ]:
# 3개의 토픽을 갖도록 LDA Modeling

topic_num = 3

ldamodel = gensim.models.ldamodel.LdaModel(corpus,
                                           num_topics=topic_num,
                                           id2word=dictionary,
                                           passes=20,
                                           iterations=50,
                                           random_state=42
                                          )

In [ ]:
# 결과 저장 리스트
c_score = [] # Coherence
p_score = [] # Perplexity
# 토픽 개수 범위 (2 ~ 9)
start_topic = 2
end_topic = 10

for i in range(start_topic, end_topic):
    # 1. LDA 모델 생성
    ldamodel = gensim.models.ldamodel.LdaModel(corpus, num_topics=i, id2word=dictionary, random_state=42)

    # 2. Coherence Score 계산
    coherencemodel = CoherenceModel(model=ldamodel, texts=df['tagged_content'], dictionary=dictionary, coherence='c_v')
    c_score.append(coherencemodel.get_coherence())

    # 3. Perplexity Score 계산
    p_score.append(ldamodel.log_perplexity(corpus))

    print(f"Topic {i} complete.")


# --- 시각화 (Coherence / Perplexity / Silhouette / Elbow) ---
x_range = range(start_topic, end_topic)

plt.figure(figsize=(24, 5))

# 1. Coherence (높을수록 좋음)
plt.subplot(1, 4, 1)
plt.plot(x_range, c_score, 'o-')
plt.title('Coherence Score')
plt.xlabel('Num Topics')
plt.ylabel('Coherence')
plt.grid(True)

# 2. Perplexity (낮을수록 좋음)
plt.subplot(1, 4, 2)
plt.plot(x_range, p_score, 'o-')
plt.title('Perplexity Score')
plt.xlabel('Num Topics')
plt.ylabel('Perplexity')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
print(c_score)
print(p_score)

In [ ]:
topic_num = 4

ldamodel = gensim.models.ldamodel.LdaModel(corpus,
                                           num_topics=topic_num,
                                           id2word=dictionary,
                                           passes=20,
                                           iterations=50,
                                           random_state=42
                                          )

In [ ]:
action_align = []

for doc in tqdm(ldamodel.get_document_topics(corpus)) :
    label = []
    value = []
    for score in doc :
        label.append(score[0])
        value.append(score[1])
    max_index = np.argmax(value)
    action_n = label[max_index]
    action_align.append(action_n)

In [ ]:
df_cluster0['action_cluster'] = action_align

In [ ]:
prepared_data = gensimvis.prepare(ldamodel, corpus, dictionary)

pyLDAvis.save_html(prepared_data, 'ldavis_cluster0_4.html')

pyLDAvis.display(prepared_data)

In [ ]:
topics = ldamodel.show_topics(num_topics=-1, formatted=True)

for topic_id, topic_desc in topics :
    print(f"Topic ID : {topic_id}, Description : {topic_desc}")

In [ ]:
0 -> 3
1 -> 4
2 -> 2
3 -> 1

In [ ]:
new_action_topic =[]

for i in tqdm(df_cluster0.action_cluster) :
    if i == 0 :
        n = 3
    elif i == 1:
        n = 4
    elif i == 2:
        n = 2
    else:
        n = 1
    new_action_topic.append(n)

In [ ]:
df_cluster0.action_cluster = new_action_topic
df_cluster0.action_cluster.value_counts()

In [ ]:
df_cluster0.info()

In [ ]:
df_cluster0 = df_cluster0[['content', 'tagged_content','cluster','action_cluster']]
df_cluster0

In [ ]:
df_cluster0.to_csv('DX_actor0_action_4.csv', encoding = 'utf-8-sig')

In [ ]:
with open('DX_actor0_action_4.pkl', 'wb') as f:
    pickle.dump(df_cluster0, f)